In [30]:
# # How to Import to Code Document
########################################
# import os, sys
# mainCodeDirectory = os.path.abspath("../..")
# path = os.path.join(mainCodeDirectory, "classes")
# sys.path.append(path)

# # Importing
# from InputData_Classes import InputData_Classes

In [10]:
# InputData_Classes
# ============================================================

# Libraries
import os
from datetime import timedelta
from glob import glob
from types import SimpleNamespace

import numpy as np
import xarray as xr; import h5py; import json
import netCDF4
import fcntl
import socket
import uuid

class InputData_Classes:

    class Model_InputData_Class:
        """
        Class for loading and reading model data for Cloud Model One (CM1),
        or any similarly-structured model, across multiple data types
        (e.g. "eulerianData", "lagrangianData") in a single call.

        dataDirectory, filePattern, coordNames : manual mode. Provide these
                        directly to load a single dataset (stored as self.data).
        pathResolverClass, resolverName, pathResolverKwargs : resolver mode.
                        pathResolverClass exposes named resolver functions
                        (see PathResolvers). The chosen resolver returns a dict:
                            { dataTypeName: {"dataDirectory": ..., "filePattern": ...,
                                              "coordNames": (...), ...}, ... }
                        Each dataTypeName becomes an attribute on this object
                        (self.eulerianData, self.lagrangianData, ...) holding its own data.
        dataTypes     : optional list to filter which dataTypeNames from the
                        resolver's dict actually get loaded (default: all).
        initialTimeLT : optional override for the initial local time (decimal
                        hours) used to build timeHoursLT. Not every model's
                        files carry this info, so it can be supplied here
                        instead — either a single number (applied to every
                        dataType) or a dict keyed by dataTypeName. Takes
                        priority over anything the resolver's config provides.
        """

        def __init__(self, dataDirectory=None, filePattern=None, coordNames=None,
                     pathResolverClass=None,
                     resolverName="CloudModelOne", pathResolverKwargs=None,
                     dataTypes=None,
                     timeCoord="time",
                     initialTimeLT=None,
                     applyDomainSubset=False,
                     metaData=None,
                     verbose=False):

            self.metaData = metaData or {}
            self.timeCoord = timeCoord
            self.applyDomainSubset = applyDomainSubset
            self.verbose = verbose
            self.pathResolverKwargs = pathResolverKwargs or {}

            for key, value in self.metaData.items():
                setattr(self, key, value)

            if pathResolverClass is None:
                pathResolverClass = InputData_Classes.PathResolvers

            [dataTypeConfig] = self.ResolvePaths(
                dataDirectory, filePattern, coordNames,
                pathResolverClass, resolverName, pathResolverKwargs)

            if dataTypes is not None:
                dataTypeConfig = {k: v for k, v in dataTypeConfig.items() if k in dataTypes}

            self.dataTypes = list(dataTypeConfig.keys())

            for dataTypeName, config in dataTypeConfig.items():
                userInitialTimeLT = (initialTimeLT.get(dataTypeName) if isinstance(initialTimeLT, dict)
                                      else initialTimeLT)
                [dataset] = self.LoadDataset(config, self.timeCoord, userInitialTimeLT, self.applyDomainSubset, self.verbose)
                setattr(self, dataTypeName, dataset)

            if self.verbose:
                self.Summary()

        # ============================================================
        # ========== Path Resolution ==========
        # ============================================================

        def ResolvePaths(self, dataDirectory, filePattern, coordNames,
                          pathResolverClass, resolverName, pathResolverKwargs):
            """
            Manual mode: if dataDirectory/filePattern given directly, wrap them
            as a single dataType named "data".
            Resolver mode: otherwise, look up resolverName in pathResolverClass
            and call it; it must return a dict of dataTypeName -> config dict.
            """
            if dataDirectory is not None and filePattern is not None:
                resolvedCoordNames = coordNames or ("time", "zf", "zh", "yf", "yh", "xf", "xh")
                dataTypeConfig = {
                    "data": {
                        "dataDirectory": dataDirectory,
                        "filePattern": filePattern,
                        "coordNames": resolvedCoordNames,
                    }
                }
                return [dataTypeConfig]

            if pathResolverClass is None or resolverName is None:
                raise ValueError(
                    "Must provide either (dataDirectory and filePattern) "
                    "or (pathResolverClass and resolverName).")

            [resolverFunction] = pathResolverClass.GetResolver(resolverName)
            pathResolverKwargs = pathResolverKwargs or {}
            [dataTypeConfig] = resolverFunction(**pathResolverKwargs)

            if not isinstance(dataTypeConfig, dict):
                raise ValueError(
                    "Resolver must return a dict mapping dataTypeName -> config dict.")

            return [dataTypeConfig]

        # ============================================================
        # ========== Per-Dataset Loading ==========
        # ============================================================

        def LoadDataset(self, config, timeCoord, userInitialTimeLT, applyDomainSubset, verbose):
            """
            Load one dataset (e.g. "eulerianData" or "lagrangianData") from its config
            dict and return it as a SimpleNamespace bundling fileList,
            coordinates, time info, grid spacing, local time, and variable names.
            """
            dataDirectory = config["dataDirectory"]
            filePattern = config["filePattern"]
            excludePatterns = config.get("excludePatterns")
            coordNames = list(config.get("coordNames", ()))
            domainSubsetBounds = config.get("domainSubsetBounds") if applyDomainSubset else None

            [fileList] = self.GetFileList(dataDirectory, filePattern, excludePatterns)
            isDataTimestepByTimestep = len(fileList) > 1

            dataset = SimpleNamespace(
                dataDirectory=dataDirectory,
                filePattern=filePattern,
                fileList=fileList,
                isDataTimestepByTimestep=isDataTimestepByTimestep,
                coordNames=coordNames,
            )

            [coordinateData] = self.GetCoordinateData(fileList, coordNames, domainSubsetBounds, verbose)
            for key, value in coordinateData.items():
                setattr(dataset, key, value)
                setattr(dataset, f"N{key}", len(np.atleast_1d(value)))

            if isDataTimestepByTimestep:
                [time] = self.GetTimeCoordinate(fileList, timeCoord)
                dataset.time = time
                dataset.Ntime = len(time)

            [dataset.timeStrings] = self.GetTimeStrings(dataset.time)

            [dataset.gridSpacing] = self.GetGridSpacing(dataset, timeCoord)

            [dataset.gridsPerKm, dataset.gridsPerMin] = self.GetGridsPerUnit(dataset.gridSpacing)

            [dataset.initialTimeLT, dataset.timeHoursModel, dataset.timeHoursLT, dataset.timeUTC] = self.GetTimeArrays(
                fileList, config, userInitialTimeLT, dataset.time)

            [dataset.varList] = self.GetVariableNames(fileList)

            dataset.domainSubsetBounds = domainSubsetBounds
            dataset.case = config.get("case")
            dataset.postprocessedRoot = os.path.join(os.path.expanduser(config.get("postprocessedDirectory") or dataDirectory),
                                                     "Postprocessed")
            [dataset.boundaryConditions, dataset.isPeriodic] = self.GetBoundaryConditions(fileList, config)
            
            dataset.OpenData = lambda file=None, **kwargs: (
                InputData_Classes.DataOpeners.OpenData(fileList, concatDim=timeCoord, file=file, 
                                                       domainSubsetBounds=domainSubsetBounds,
                                                       **kwargs)[0])
            dataset.SaveH5 = lambda data, file, label, **kwargs: (
                InputData_Classes.DataExporters.SaveH5(dataset, data, file, label, **kwargs))
            dataset.LoadH5 = lambda label, file=0, varNames=None: (
                InputData_Classes.DataExporters.LoadH5(dataset, label, file, varNames))
            return [dataset]

        # ============================================================
        # ========== Data Discovery ==========
        # ============================================================

        def GetFileList(self, dataDirectory, filePattern, excludePatterns=None):
            """Find all files matching filePattern in dataDirectory, sorted,
            excluding any that also match excludePatterns (e.g. other CM1 output
            types like pdata/stats that happen to share the same glob prefix)."""
            fileList = sorted(glob(os.path.join(dataDirectory, filePattern)))

            if excludePatterns:
                [fileList] = self.FilterExcluded(fileList, dataDirectory, excludePatterns)

            if not fileList:
                raise FileNotFoundError(
                    f"No files found in {dataDirectory} matching {filePattern}"
                    + (f" (after excluding {excludePatterns})" if excludePatterns else ""))
            return [fileList]

        def FilterExcluded(self, fileList, dataDirectory, excludePatterns):
            """Helper: drop any files matching excludePatterns from fileList."""
            excludedFiles = set()
            for pattern in excludePatterns:
                excludedFiles.update(glob(os.path.join(dataDirectory, pattern)))
            filtered = [f for f in fileList if f not in excludedFiles]
            return [filtered]

        # ============================================================
        # ========== Data Loading Functions ==========
        # ============================================================

        def GetCoordinateData(self, fileList, coordNames, domainSubsetBounds, verbose):
            """
            Extract the requested coordinate arrays from the FIRST file.
            Coordinates listed in coordNames but missing from the file are
            skipped (with a note if verbose).
            """
            extracted = {}
            with xr.open_dataset(fileList[0], decode_timedelta=True) as ds:
                ds = InputData_Classes.DataOpeners.ApplyDomainSubset(ds, domainSubsetBounds)
                for k in coordNames:
                    if k in ds:
                        extracted[k] = ds[k].values
                    elif verbose:
                        print(f"Note: coordinate '{k}' not found in {fileList[0]}; skipping.")
            return [extracted]

        def GetTimeCoordinate(self, fileList, timeCoord, method="constructed", dtSeconds=None):
            """
            Time coordinate (timedelta64[ns]), one entry per file.
            method : "xarray" | "netCDF4" (read every file) |
                     "constructed" (t0 + arange(N)*dt; checked against the last file)
            """
            def Read(f):
                with netCDF4.Dataset(f, "r") as nc:
                    return float(np.atleast_1d(nc.variables[timeCoord][:])[0])

            if method == "xarray":
                seconds = [xr.open_dataset(f, decode_timedelta=True)[timeCoord].values.ravel()[0]
                           / np.timedelta64(1, "s") for f in fileList]
            elif method == "netCDF4":
                seconds = [Read(f) for f in fileList]
            elif method == "constructed":
                t0 = Read(fileList[0])
                dt = Read(fileList[1]) - t0;
                seconds = t0 + np.arange(len(fileList)) * dt
                if not np.isclose(seconds[-1], Read(fileList[-1])):
                    raise ValueError("Files missing or unevenly spaced; use method='netCDF4'.")
            else:
                raise ValueError(f"Unknown method '{method}'")

            return [(np.asarray(seconds) * 1e9).astype("timedelta64[ns]")]

        def GetTimeStrings(self, times):
            """
            Convert the time coordinate into filesystem-safe strings
            (e.g. "0-05-00" for 5 minutes).
            """
            times = np.asarray(times)
            if np.issubdtype(times.dtype, np.timedelta64):
                seconds = times / np.timedelta64(1, "s")
            else:
                seconds = times / 1e9  # assume nanoseconds
            timeStrings = [str(timedelta(seconds=float(s))).replace(":", "-") for s in seconds]
            return [timeStrings]

        def GetGridSpacing(self, dataset, timeCoord):
            """
            Compute spacing for every coordinate on dataset. Returns a
            SimpleNamespace with attributes named "d" + coordName (e.g.
            "dtime", "dzf", "dxh").
            """
            gridSpacing = SimpleNamespace()

            for coordName in dataset.coordNames:
                if not hasattr(dataset, coordName):
                    continue
                values = getattr(dataset, coordName)
                if len(np.atleast_1d(values)) <= 1:
                    continue

                if coordName == timeCoord:
                    spacing = self.GetTimeSpacing(values)
                else:
                    spacing = self.GetSpatialSpacing(values)

                setattr(gridSpacing, f"d{coordName}", spacing)

            return [gridSpacing]
        
        def GetGridsPerUnit(self, gridSpacing):
            """
            gridsPerKm: grid cells per km in x (from dxh). gridsPerMin: output steps per minute (from dtime).

            Useful if for example wanting to determine how many gridboxes 
            are a radius of a (certain number of kms) * gridsPerKms.
            """
            if not (hasattr(gridSpacing, "dxh") and hasattr(gridSpacing, "dtime")):
                return [None, None]
        
            gridsPerKm = 1 / (gridSpacing.dxh / 1000)
            gridsPerMin = 1 / (gridSpacing.dtime / 60)
            return [gridsPerKm, gridsPerMin]

        def GetTimeSpacing(self, values):
            """Helper: spacing (s) for a time coordinate, scalar or diff array."""
            t = np.asarray(values)
            if np.issubdtype(t.dtype, np.timedelta64):
                diffs = np.diff(t) / np.timedelta64(1, "s")
            else:
                diffs = np.diff(t) / 1e9  # assume nanoseconds
            return self.CollapseIfUniform(diffs)

        def GetSpatialSpacing(self, values):
            """Helper: spacing (m) for a spatial coordinate, scalar or diff array.
            Assumes km input, per CM1 convention. Rounded to 0.1 m to remove float32 noise in
            the stored coordinates (a few cm at ~300 km), so e.g. 400 m comes out as exactly 400.0."""
            diffs = np.round(np.diff(np.asarray(values, dtype=np.float64)) * 1000, 1)
            return self.CollapseIfUniform(diffs)

        def CollapseIfUniform(self, diffs):
            """Helper: return a single scalar if all diffs are equal (evenly-spaced
            coordinate, e.g. xh/yh), otherwise the full diff array (unevenly-spaced,
            e.g. staggered zf levels)."""
            if np.allclose(diffs, diffs[0]):
                return diffs[0].item()
            return diffs

        def GetBoundaryConditions(self, fileList, config):
            """
            Lateral boundary conditions from the first file's global attributes, using the resolver's
            boundaryConditionAttrs ({"west": attrName, ...}) and boundaryConditionCodes ({code: name}).
            Returns [boundaryConditions, isPeriodic]:
              boundaryConditions : {"west": "periodic", "east": "open-radiative", ...} (None if not available)
              isPeriodic         : {"x": bool, "y": bool}, True only if both sides of that axis are periodic
            """
            attrNames = config.get("boundaryConditionAttrs")
            codes = config.get("boundaryConditionCodes", {})
            if not attrNames:
                return [None, None]
            with xr.open_dataset(fileList[0], decode_timedelta=True) as ds:
                boundaryConditions = {side: codes.get(int(ds.attrs[name]), f"code {int(ds.attrs[name])}")
                                      if name in ds.attrs else None
                                      for side, name in attrNames.items()}
            isPeriodic = {"x": boundaryConditions.get("west") == "periodic" and boundaryConditions.get("east") == "periodic",
                          "y": boundaryConditions.get("south") == "periodic" and boundaryConditions.get("north") == "periodic"}
            return [boundaryConditions, isPeriodic]

        def GetTimeArrays(self, fileList, config, userInitialTimeLT, time):
            """
            Time arrays built from the actual time values, so they are right for a run that does not
            start at t = 0 (e.g. a run restarted from another run's restart file):
              timeHoursModel : hours since the model start time (same clock in every run)
              timeHoursLT    : local time in hours, initialTimeLT + timeHoursModel. Not wrapped at 24,
                               so noon on day 2 is 36.0
              timeUTC        : datetime64 in UTC (None if the file has no year/month/day attributes)
            initialTimeLT (decimal hours, the local time at the model start) is found in this order:
              1. userInitialTimeLT, passed by the caller
              2. config["initialTimeLT"], hardcoded by the resolver
              3. config["initialTimeAttrs"]: the file's hour/minute attributes (UTC) + UTC_offset
            """
            initialTimeAttrs = config.get("initialTimeAttrs") or {}
            with xr.open_dataset(fileList[0], decode_timedelta=True) as ds:
                attrs = dict(ds.attrs)

            #Seconds since the model start, from the file times
            [startUTC] = self.GetStartUTC(attrs, initialTimeAttrs)
            [seconds] = self.TimeToSeconds(time, startUTC)
            timeHoursModel = seconds / 3600

            #Local time at the model start
            if userInitialTimeLT is not None:
                initialTimeLT = userInitialTimeLT
            elif "initialTimeLT" in config:
                initialTimeLT = config["initialTimeLT"]
            elif initialTimeAttrs:
                hour = attrs.get(initialTimeAttrs.get("hour"), 0)
                minute = attrs.get(initialTimeAttrs.get("minute"), 0)
                initialTimeLT = (hour + minute / 60 + initialTimeAttrs.get("UTC_offset", 0)) % 24
            else:
                initialTimeLT = None

            timeHoursLT = None if initialTimeLT is None else initialTimeLT + timeHoursModel
            timeUTC = None if startUTC is None else startUTC + (seconds * 1e9).astype("timedelta64[ns]")
            return [initialTimeLT, timeHoursModel, timeHoursLT, timeUTC]

        def GetStartUTC(self, attrs, initialTimeAttrs):
            """Helper: model start time (UTC) from the file's year/month/day/hour/minute attributes,
            or None if the date attributes are missing. Attribute names default to the key names."""
            names = {k: initialTimeAttrs.get(k, k) for k in ("year", "month", "day", "hour", "minute")}
            if not all(names[k] in attrs for k in ("year", "month", "day")):
                return [None]
            v = {k: int(attrs.get(names[k], 0)) for k in names}
            return [np.datetime64(f"{v['year']:04d}-{v['month']:02d}-{v['day']:02d}T{v['hour']:02d}:{v['minute']:02d}", "ns")]

        def TimeToSeconds(self, time, startUTC):
            """Helper: the time coordinate as float seconds since the model start, whether xarray
            decoded it as timedelta64, as datetime64, or left it as plain seconds."""
            t = np.asarray(time)
            if np.issubdtype(t.dtype, np.timedelta64):
                return [t / np.timedelta64(1, "s")]
            if np.issubdtype(t.dtype, np.datetime64):
                if startUTC is None:
                    raise ValueError("Time is datetime64 but the file has no start date attributes.")
                return [(t - startUTC) / np.timedelta64(1, "s")]
            return [t.astype(float)]

        def GetVariableNames(self, fileList):
            """Get list of data-variable names from the first file."""
            with xr.open_dataset(fileList[0], decode_timedelta=True) as ds:
                varList = list(ds.data_vars)
            return [varList]


        # ============================================================
        # === Information ========================================
        # ============================================================

        def Summary(self):
            """Print a summary of every loaded dataset."""
            print("=== Model Input Data Summary ===")
            for key, value in self.metaData.items():
                print(f" {key}: {value}")
            for i, dataTypeName in enumerate(self.dataTypes):
                dataset = getattr(self, dataTypeName)
                self.PrintDatasetSummary(dataTypeName, dataset)
                if i < len(self.dataTypes) - 1:
                    print()
            print("=" * 32, "\n")

        def PrintDatasetSummary(self, dataTypeName, dataset):
            """Helper: print one dataset's block within the overall Summary."""
            print(f" --- {dataTypeName} ---")
            print(f"  dataDirectory: {dataset.dataDirectory}")
            print(f"  postprocessed: {dataset.postprocessedRoot}")
            print(f"  filePattern:   {dataset.filePattern}")
            print(f"  # files:       {len(dataset.fileList)}")
            print(f"  timestep-by-timestep: {dataset.isDataTimestepByTimestep}")
            print(f"  Time steps:    {len(np.atleast_1d(dataset.time))}")
            print(f"  Initial LT:    {dataset.initialTimeLT}")
            print(f"  Model hours:   {dataset.timeHoursModel[0]:.3f} to {dataset.timeHoursModel[-1]:.3f}")
            if dataset.timeUTC is not None:
                print(f"  UTC:           {dataset.timeUTC[0]} to {dataset.timeUTC[-1]}")
            print(f"  Variables:     {dataset.varList}")

    class PathResolvers:
        """Namespace of named path-resolver functions, selectable by string key.
        Each resolver returns a dict: dataTypeName -> config dict, with keys
        "dataDirectory", "filePattern", "coordNames", and optionally
        "excludePatterns", "initialTimeLT", "initialTimeAttrs",
        "postprocessedDirectory" (folder that holds Postprocessed/; default dataDirectory).
        """

        #Add Per-simulation Settings Here
        SIMULATIONS = {
            "Test": {"folder": "Simulation_Test", "case": "TRACER", "UTC_offset": -5, "domainSubsetBounds": {}}, #testing
            "1":    {"folder": "Simulation_1",    "case": "TRACER", "UTC_offset": -5, "domainSubsetBounds": {}}, #unsuccessful
            "2":    {"folder": "Simulation_2",    "case": "DYNAMO", "UTC_offset": +5, "domainSubsetBounds": {},
                     "postprocessedDirectory": "/home/air673/koa_scratch/3.Moist-Preconditioning-Project/Simulation_2/"}, #control
            "3":    {"folder": "Simulation_3",    "case": "DYNAMO", "UTC_offset": +5, "domainSubsetBounds": {}}, #forced
        }

        @staticmethod
        def CloudModelOne(inputDirectory, simulationLabel, **overrides):
            sim = InputData_Classes.PathResolvers.SIMULATIONS[str(simulationLabel)]
            dataDirectory = os.path.join(inputDirectory, "Model", sim["folder"])
            timeAttrs = {"hour": "hour", "minute": "minute", "UTC_offset": sim["UTC_offset"]}
            CM1_BOUNDARY_CONDITIONS = {1: "periodic", 2: "open-radiative", 3: "rigidwall-free-slip", 4: "rigidwall-no-slip"}

            dataTypeConfig = {
                "eulerianData": {
                    "dataDirectory": dataDirectory,
                    "filePattern": "cm1out_*.nc",
                    "excludePatterns": ("cm1out_pdata*.nc", "cm1out_stats*.nc"),
                    "coordNames": ("time", "zf", "zh", "yf", "yh", "xf", "xh"),
                    "boundaryConditionAttrs": {"west": "wbc", "east": "ebc", "south": "sbc", "north": "nbc"},
                    "boundaryConditionCodes": CM1_BOUNDARY_CONDITIONS,
                    "initialTimeAttrs": timeAttrs,
                    "domainSubsetBounds": sim["domainSubsetBounds"], #e.g. {"xh": slice(20, 280)}
                    "case": sim["case"],
                    "postprocessedDirectory": sim.get("postprocessedDirectory"),
                },
                "lagrangianData": {
                    "dataDirectory": dataDirectory,
                    "filePattern": "cm1out_pdata.nc",
                    "coordNames": ("time",),
                    "initialTimeAttrs": timeAttrs,
                    "case": sim["case"],
                    "postprocessedDirectory": sim.get("postprocessedDirectory"),
                },
            }

            dataTypeConfig["eulerianData"].update(overrides)
            return [dataTypeConfig]

        # Add more resolvers here as needed, e.g.:
        # @staticmethod
        # def MPAS(inputDirectory, runLabel):
        #     ...

        @classmethod
        def GetRegistry(self):
            """Maps resolver name (string) -> resolver function."""
            registry = {
                "CloudModelOne": self.CloudModelOne,
                # "MPAS": self.MPAS,
            }
            return [registry]

        @classmethod
        def GetResolver(self, resolverName):
            """Look up a resolver function by name, with a helpful error if missing."""
            [registry] = self.GetRegistry()
            if resolverName not in registry:
                raise KeyError(
                    f"Unknown resolverName '{resolverName}'. Options: {list(registry)}")
            return [registry[resolverName]]

    class DataOpeners:
        """Namespace of stateless functions for opening netCDF file(s) into
        an xarray Dataset, given a file list and optional file-index selection."""

        @staticmethod
        def ApplyDomainSubset(ds, domainSubsetBounds):
            """
            Crop ds to domainSubsetBounds ({dim: slice(lo, hi)}, inclusive, in the
            coordinate's own units).

            Staggered grids are cropped together: bounds on a cell-center dimension
            (xh/yh/zh) also crop the matching face dimension (xf/yf/zf) to the N+1
            faces bounding those cells, and bounds on a face dimension crop the
            matching centers to the cells between those faces. If both are given
            explicitly, each is applied as written.
            """
            if not domainSubsetBounds:
                return ds

            def SelectedIndices(dim, bound):
                values = ds[dim].values
                lo = -np.inf if bound.start is None else bound.start
                hi = np.inf if bound.stop is None else bound.stop
                idx = np.flatnonzero((values >= lo) & (values <= hi))
                if idx.size == 0:
                    raise ValueError(f"domainSubsetBounds for '{dim}' ({bound}) selects nothing.")
                return idx[0], idx[-1]

            indexers = {}
            for dim, bound in domainSubsetBounds.items():
                if dim not in ds.dims:
                    continue
                [first, last] = SelectedIndices(dim, bound)
                indexers[dim] = slice(first, last + 1)

                pair = {'h': 'f', 'f': 'h'}.get(dim[-1])
                other = dim[:-1] + pair if pair else None
                if other is None or other in domainSubsetBounds or other not in ds.dims:
                    continue                                   # no partner, or user set it explicitly

                nFaces, nCenters = (ds.sizes[other], ds.sizes[dim]) if dim.endswith('h') else (ds.sizes[dim], ds.sizes[other])
                if nFaces != nCenters + 1:
                    raise ValueError(f"'{dim}' and '{other}' aren't a staggered pair (expected N+1 faces, got {nFaces} vs {nCenters}).")

                indexers[other] = slice(first, last + 2) if dim.endswith('h') else slice(first, last)
            return ds.isel(indexers)
    
        @staticmethod
        def OpenData(fileList, concatDim="time", file=None, domainSubsetBounds=None, **kwargs):
            """
            Open one or more netCDF files as a single xarray Dataset.
            ***Caller is responsible for closing [.close()] and deleting (del) the returned Dataset for IO efficiency
            (i.e. opening many files in a loop).***
            """
            fileList = [fileList] if isinstance(fileList, str) else list(fileList)
            [selectedFiles] = InputData_Classes.DataOpeners.SelectFiles(fileList, file)
        
            if len(selectedFiles) == 1:
                [ds] = InputData_Classes.DataOpeners.OpenSingleFile(
                    selectedFiles[0], domainSubsetBounds=domainSubsetBounds, **kwargs)
            else:
                [ds] = InputData_Classes.DataOpeners.OpenMultipleFiles(
                    selectedFiles, concatDim, domainSubsetBounds=domainSubsetBounds, **kwargs)
        
            return [ds]
    
        @staticmethod
        def SelectFiles(fileList, file):
            """Helper: resolve the file selector into the actual subset of
            fileList to open."""
            if file is None:
                return [fileList]
            if isinstance(file, int):
                return [[fileList[file]]]
            indices = list(file)
            return [[fileList[i] for i in indices]]
            
        @staticmethod
        def OpenSingleFile(filePath, domainSubsetBounds=None, **kwargs):
            ds = xr.open_dataset(filePath, decode_timedelta=True, **kwargs)
            ds = InputData_Classes.DataOpeners.ApplyDomainSubset(ds, domainSubsetBounds)
            return [ds]

        @staticmethod
        def OpenMultipleFiles(fileList, concatDim, domainSubsetBounds=None, **kwargs):
            ds = xr.open_mfdataset(
                fileList, concat_dim=concatDim, combine="nested",
                decode_timedelta=True, **kwargs)
            ds = InputData_Classes.DataOpeners.ApplyDomainSubset(ds, domainSubsetBounds)
            return [ds]
            
    class DataExporters:
        """Namespace of stateless functions for exporting an already-prepared
        xarray Dataset to disk as HDF5. Each file's path is fixed by its label and file index
        (<postprocessedRoot>/<label>/cm1_<number>_<label>.h5), so files are found from that
        naming convention alone. postprocessedRoot is <dataDirectory>/Postprocessed, or
        <postprocessedDirectory>/Postprocessed when the resolver sets postprocessedDirectory (e.g. on scratch).
        A shared JSON manifest (<postprocessedRoot>/manifest.json) is also kept as a
        record of what has been saved, but nothing depends on it being complete."""
    
        @staticmethod
        def SaveH5(dataset, data, file, label, compression=None, verbose=False, updateManifest=True, **kwargs):
            """
            Save data (an already-prepared xr.Dataset) as HDF5 into <dataset.postprocessedRoot>/<label>/.
            The file is written under a temporary name and renamed when complete, so a file with the final name
            always holds complete data (a job killed mid-write leaves only a temporary file).
            updateManifest: also record the file in the shared manifest.json. Pass False from parallel job arrays
            and run RebuildManifest once afterwards, so jobs never write the shared file at the same time.
            """
            postprocessedRoot = dataset.postprocessedRoot
            postprocessedDir = os.path.join(postprocessedRoot, label)
            os.makedirs(postprocessedDir, exist_ok=True)
            [outputFile] = InputData_Classes.DataExporters.GetH5FileName(dataset, label, file)
            outputPath = os.path.join(postprocessedDir, outputFile)

            #Write to a unique temporary file, then rename it to the final name in one step
            [tmpPath] = InputData_Classes.DataExporters.GetTemporaryPath(outputPath)
            try:
                with h5py.File(tmpPath, "w") as f:
                    for name, da in data.data_vars.items():
                        [values] = InputData_Classes.DataExporters.ToH5Compatible(da.values)
                        comp = compression if values.ndim > 0 else None
                        f.create_dataset(name, data=values, compression=comp, **kwargs)
                    for name, coord in data.coords.items():
                        [values] = InputData_Classes.DataExporters.ToH5Compatible(coord.values)
                        comp = compression if values.ndim > 0 else None
                        f.create_dataset(name, data=values, compression=comp, **kwargs)
                os.replace(tmpPath, outputPath)
            finally:
                #Remove the temporary file if anything failed before the rename
                if os.path.exists(tmpPath):
                    os.remove(tmpPath)

            if updateManifest:
                InputData_Classes.DataExporters.AddManifestEntry(dataset, label, file, os.path.join(label, outputFile))

            if verbose:
                print(f"Saved: {outputPath}")
            return [outputPath]
        
        @staticmethod
        def ToH5Compatible(values):
            """
            Helper: h5py can't store numpy timedelta64/datetime64 dtypes directly.
            Convert timedelta64 arrays to float64 seconds, and datetime64 arrays
            to int64 nanoseconds-since-epoch, so they can be written as an HDF5
            dataset. Any other dtype is passed through unchanged.
            """
            values = np.asarray(values)
            if np.issubdtype(values.dtype, np.timedelta64):
                values = values / np.timedelta64(1, "s")
            elif np.issubdtype(values.dtype, np.datetime64):
                values = values.astype("datetime64[ns]").astype(np.int64)
            return [values]
                        
        @staticmethod
        def ReadH5File(path, varNames=None):
            """Helper: read one HDF5 file into a dict of {name: array}.
            varNames, if given, restricts which datasets are actually read."""
            data = {}
            with h5py.File(path, "r") as f:
                keys = varNames if varNames is not None else list(f.keys())
                for name in keys:
                    data[name] = f[name][()]
            return [data]
        
        @staticmethod
        def LoadH5(dataset, label, file=0, varNames=None):
            """
            Read back HDF5 file(s) saved via SaveH5 under this label. Paths come from the naming convention, so the
            shared manifest is not read (no lock contention, and a missing manifest entry cannot break loading).
            file : int -> ({varName: array}, filePath)
            file : list/tuple of int -> ({fileIndex: {varName: array}}, {fileIndex: filePath})
            varNames : optional list -- only read these variables instead of every
                       dataset in the file (saves I/O when you only need a few).
            """
            if isinstance(file, int):
                [outputPath] = InputData_Classes.DataExporters.GetH5Path(dataset, label, file)
                [data] = InputData_Classes.DataExporters.ReadH5File(outputPath, varNames)
                return [data, outputPath]

            if isinstance(file, (list, tuple)):
                result = {}
                paths = {}
                for f in file:
                    [outputPath] = InputData_Classes.DataExporters.GetH5Path(dataset, label, f)
                    [result[f]] = InputData_Classes.DataExporters.ReadH5File(outputPath, varNames)
                    paths[f] = outputPath
                return [result, paths]

            raise TypeError(f"file must be int or list/tuple of int, got {type(file).__name__}")

        @staticmethod
        def GetH5FileName(dataset, label, file):
            """Helper: file name for this label and file index (cm1_<number>_<label>.h5), where <number> is the
            number in the source file name (timestep-by-timestep data) or the zero-padded file index."""
            if dataset.isDataTimestepByTimestep:
                number = os.path.splitext(os.path.basename(dataset.fileList[file]))[0].split("_")[-1]
            else:
                number = f"{file:06d}"
            return [f"cm1_{number}_{label}.h5"]

        @staticmethod
        def GetH5Path(dataset, label, file):
            """Helper: full path of the saved H5 file for this label and file index; a clear error if it does not
            exist (not yet computed, or the job that should have written it failed)."""
            [fileName] = InputData_Classes.DataExporters.GetH5FileName(dataset, label, file)
            outputPath = os.path.join(dataset.postprocessedRoot, label, fileName)
            if not os.path.exists(outputPath):
                raise FileNotFoundError(f"No '{label}' file for file index {file}: {outputPath}")
            return [outputPath]

        @staticmethod
        def GetTemporaryPath(path):
            """Helper: a temporary file name next to path, unique across nodes and processes (host name, process id
            and a random suffix), so two jobs can never write the same temporary file."""
            return [f"{path}.{socket.gethostname()}.{os.getpid()}.{uuid.uuid4().hex[:8]}.tmp"]

        @staticmethod
        def ReadManifestFile(manifestPath):
            """Helper: read manifest.json; an empty dict if it doesn't exist or can't be parsed."""
            try:
                with open(manifestPath, "r") as f:
                    return [json.load(f)]
            except (FileNotFoundError, json.JSONDecodeError):
                return [{}]

        @staticmethod
        def LoadManifest(manifestPath):
            """Helper: load manifest.json (a record of saved files), or an empty dict if it doesn't exist yet."""
            lockPath = manifestPath + ".lock"
            with open(lockPath, "w") as lockFile:
                fcntl.flock(lockFile, fcntl.LOCK_SH)
                try:
                    return InputData_Classes.DataExporters.ReadManifestFile(manifestPath)
                finally:
                    fcntl.flock(lockFile, fcntl.LOCK_UN)
        
        @staticmethod
        def SaveManifest(manifestPath, manifest):
            """Helper: write manifest.json atomically (to a unique temporary file, then renamed over the old one),
            so a reader never sees a half-written or empty file."""
            [tmpPath] = InputData_Classes.DataExporters.GetTemporaryPath(manifestPath)
            try:
                with open(tmpPath, "w") as f:
                    json.dump(manifest, f, indent=2)
                    f.flush()
                    os.fsync(f.fileno())
                os.replace(tmpPath, manifestPath)
            finally:
                if os.path.exists(tmpPath):
                    os.remove(tmpPath)

        @staticmethod
        def AddManifestEntry(dataset, label, file, relativePath):
            """Helper: add one entry to manifest.json under the lock. Where locks are not shared across nodes, an
            entry can still be lost when two jobs update at once; that only affects this record, since loading uses
            the naming convention, and RebuildManifest restores it."""
            manifestPath = os.path.join(dataset.postprocessedRoot, "manifest.json")
            lockPath = manifestPath + ".lock"
            with open(lockPath, "w") as lockFile:
                fcntl.flock(lockFile, fcntl.LOCK_EX)
                try:
                    [manifest] = InputData_Classes.DataExporters.ReadManifestFile(manifestPath)
                    manifest.setdefault(label, {})[str(file)] = relativePath
                    InputData_Classes.DataExporters.SaveManifest(manifestPath, manifest)
                finally:
                    fcntl.flock(lockFile, fcntl.LOCK_UN)

        @staticmethod
        def RebuildManifest(dataset, removeTemporary=False, checkReadable=False):
            """
            Rebuild <postprocessedRoot>/manifest.json after a job array run with updateManifest=False. 
            Prints, per label, the number of files found and the
            file indices that are missing, and lists leftover temporary files from interrupted writes
            (removeTemporary=True deletes them).
            checkReadable: also open every file and count it as missing if it can't be read (e.g. a file 
            written directly to its final name by an older version of SaveH5 and cut short by a crash or time 
            limit).
            """
            postprocessedRoot = dataset.postprocessedRoot
            nFiles = len(dataset.fileList) if dataset.isDataTimestepByTimestep else dataset.Ntime
            manifest, missing = {}, {}
            for label in sorted(os.listdir(postprocessedRoot)):
                labelDir = os.path.join(postprocessedRoot, label)
                if not os.path.isdir(labelDir):
                    continue
                present = set(os.listdir(labelDir))
                for file in range(nFiles):
                    [fileName] = InputData_Classes.DataExporters.GetH5FileName(dataset, label, file)
                    readable = fileName in present
                    if readable and checkReadable:
                        try:
                            with h5py.File(os.path.join(labelDir, fileName), "r") as f:
                                list(f.keys())
                        except OSError:
                            readable = False
                            print(f"{label}: unreadable file {fileName}")
                    if readable:
                        manifest.setdefault(label, {})[str(file)] = os.path.join(label, fileName)
                    else:
                        missing.setdefault(label, []).append(file)
                #Leftover temporary files from writes that were interrupted
                leftovers = [name for name in present if name.endswith(".tmp")]
                if leftovers:
                    print(f"{label}: {len(leftovers)} leftover temporary files"
                          + (" (removed)" if removeTemporary else ""))
                    if removeTemporary:
                        for name in leftovers:
                            os.remove(os.path.join(labelDir, name))
            manifestPath = os.path.join(postprocessedRoot, "manifest.json")
            lockPath = manifestPath + ".lock"
            with open(lockPath, "w") as lockFile:
                fcntl.flock(lockFile, fcntl.LOCK_EX)
                try:
                    InputData_Classes.DataExporters.SaveManifest(manifestPath, manifest)
                finally:
                    fcntl.flock(lockFile, fcntl.LOCK_UN)
            for label in sorted(set(manifest) | set(missing)):
                nMissing = len(missing.get(label, []))
                print(f"{label}: {len(manifest.get(label, {}))} of {nFiles} files"
                      + (f"; missing file indices {missing[label]}" if nMissing else ""))
            return [manifest, missing]

    class UtilityFunctions:
    
        @staticmethod
        def CalculateTimeIndex(times, time_hr, tolerance_hr=None):
            """Index of the time nearest time_hr (same units as times, e.g. timeHoursLT or timeHoursModel).
            Raises an error if the nearest time is more than one output step away (default), i.e. the
            requested time is outside the run, instead of silently returning the first or last index."""
            times = np.asarray(times)
            i = int(np.abs(times - time_hr).argmin())
            if tolerance_hr is None:
                tolerance_hr = np.median(np.diff(times)) if len(times) > 1 else 0.0
            if abs(times[i] - time_hr) > tolerance_hr:
                raise ValueError(f"Requested time {time_hr} is outside the data ({times[0]:.3f} to {times[-1]:.3f}).")
            return i

        @staticmethod
        def ConvertLagrangianSpatialVariablesToIndex(z,y,x,eulerianData,invalid=-100):
            zf=eulerianData.zf*1e3; Z=np.searchsorted(zf,z)-1
            yf=eulerianData.yf*1e3; Y=np.searchsorted(yf,y)-1
            xf=eulerianData.xf*1e3; X=np.searchsorted(xf,x)-1
            VALID = (Z>=0)&(Z<eulerianData.Nzh)&(Y>=0)&(Y<eulerianData.Nyh)&(X>=0)&(X<eulerianData.Nxh)
            [Z,Y,X] = [np.where(VALID,idx,invalid).astype(np.int16) for idx in (Z,Y,X)]
            return [Z,Y,X,VALID]

# #--------------------------------------------------
# #Example Loading
# #--------------------------------------------------
# #Load Classes
# mainCodeDirectory = os.path.abspath("/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/3.Moist-Preconditioning-Project/Coding_Directory/Code")
# classDirectory = os.path.join(mainCodeDirectory, "classes")
# sys.path.append(classDirectory)

# from OutputData_Classes import OutputData_Classes
# from InputData_Classes import InputData_Classes

# #Loading DatabaseManager
# DatabaseManager\
# =OutputData_Classes.DatabaseManager_Class(classDirectory=classDirectory,
#                                           inputDirectory="../../Input/", outputDirectory="../../Output/",
#                                           verbose=False)

# InputData = InputData_Classes.Model_InputData_Class(pathResolverKwargs={"inputDirectory": DatabaseManager.inputDirectory, 
#                                                                         "simulationLabel": "1"},applyDomainSubset=True)